# process data

- adding 20 & 50 day simple moving Average (SMA) 
- and relative strength index (RSI) values
- will meaningfully enhance my data set

In [1]:
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

original_data = os.getenv('FINAL_ALIGNED_DATA_CSV')

# set the Date as the index
try:
    df = pd.read_csv(original_data, index_col='Date', parse_dates=True)
except FileNotFoundError:
    print("aligned.csv not found, fix it")
    exit()

# calculate simple moving averages (SMA) for NVDA
df['SMA_20_NVDA'] = df['Close_NVDA'].rolling(window=20).mean()
df['SMA_50_NVDA'] = df['Close_NVDA'].rolling(window=50).mean()

# calculate relative strength index (RSI) for NVDA
# RSI formula derived from https://www.macroption.com/rsi-calculation/
delta = df['Close_NVDA'].diff(1)
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

# use exponential moving average for RSI calculation
avg_gain = gain.ewm(com=13, adjust=False).mean()
avg_loss = loss.ewm(com=13, adjust=False).mean()

rs = avg_gain / avg_loss
df['RSI_NVDA'] = 100 - (100 / (1 + rs))

# clean and save
# calculations above create empty values (NaN) for the first 50 rows
# drop these rows because the model can't work with missing data
df.dropna(inplace=True)

# save the enriched dataset to a new file
df.to_csv('data_with_features.csv')


print(f"new dataset has {len(df)} rows after dropping initial empty values")
print("saved to data_with_features.csv")
print("\nfirst 5 rows of the new dataset")
print(df.head())

new dataset has 6668 rows after dropping initial empty values
saved to data_with_features.csv

first 5 rows of the new dataset
            Open_NVDA  High_NVDA  Low_NVDA  Close_NVDA  Volume_NVDA  Open_NDX  \
Date                                                                            
1999-04-05   0.038744   0.039668  0.037839    0.038058     27293895   2146.13   
1999-04-06   0.038287   0.038512  0.036230    0.036919     19377868   2219.64   
1999-04-07   0.037377   0.040352  0.036919    0.040127     24781851   2218.83   
1999-04-08   0.040352   0.041963  0.039894    0.040811     35550119   2192.29   
1999-04-09   0.041039   0.041039  0.039894    0.040127     13345989   2224.75   

            High_NDX  Low_NDX  Close_NDX   Volume_NDX  ...  Low_SPX  \
Date                                                   ...            
1999-04-05   2219.97  2146.13    2219.64  168606897.0  ...  1293.72   
1999-04-06   2244.25  2200.56    2218.83  192067241.0  ...  1311.07   
1999-04-07   2251.29 